# Catalogue extraction: OCR + local LLM

Turns a scanned exhibition catalogue (PDF) into a CSV of the artworks in it, using the project's three data models: [Item in Exhibition](../../data-models/item-in-exhibition-datamodel.qmd), [Artist](../../data-models/artist-datamodel.qmd) and [Exhibition](../../data-models/exhibition-datamodel.qmd). The columns are the same as those of the regex pipeline in [`catalogue-ocr.ipynb`](../catalogue-ocr.ipynb) (`scripts/extract_artworks.py`), and the output is checked with the same validator (`scripts/validate_csv.py`).

**How it works**

1. **OCR**: Tesseract reads the page images and writes plain text plus hOCR (text with page positions).
2. **Chunking**: the text is cut into overlapping chunks of about 800 words, with page numbers tracked.
3. **Extraction**: a local LLM (via [Ollama](https://ollama.com)) finds the catalogue entries in each chunk and returns them as JSON. Every chunk and every raw reply is saved, so a missed or wrong entry can be traced back to the exact text the model saw.
4. **Clean-up**: duplicates from the chunk overlap are merged, each entry is matched to its PDF page, and artist details are looked up on Wikidata.
5. **Output and validation**: the data-model CSV and a QuickStatements CSV are written. Rows that need a human check are marked `[?]` in `notes`.
6. **Review (optional)**: a browser app shows each entry next to its page, with the extracted values boxed, for correcting, adding or deleting entries.

Nothing is sent to an external AI service. The only network access is to Wikidata, for artist details, and that can be turned off.

**To run it on a new catalogue:** check the setup (below), edit the **Parameters** cell, then *Run All*. If a run is interrupted, run it again: OCR results and finished LLM chunks are reused.

**Setup (once per computer)**

| What | Linux (Debian/Ubuntu) | macOS (Homebrew) |
|---|---|---|
| Tesseract + language data | `sudo apt install tesseract-ocr tesseract-ocr-deu` | `brew install tesseract tesseract-lang` |
| PDF to image (`pdftoppm`) | `sudo apt install poppler-utils` | `brew install poppler` |
| Ollama + model | install from [ollama.com](https://ollama.com), then `ollama pull qwen2.5:14b` | same |
| Spell check (optional, for the OCR quality score) | `sudo apt install aspell aspell-en aspell-de` | `brew install aspell` |
| Python packages | `pip install beautifulsoup4 lxml` | same |

`qwen2.5:14b` needs about 10 GB of GPU memory (or a Mac with 16 GB+ unified memory). On smaller machines use `qwen2.5:7b`, which is faster but misses more entries.

The notebook must be run from inside the repository, because it imports the data-model columns and the validator from `scripts/`.

## Parameters

Edit this cell for each catalogue. Everything else is derived from it. The values below are for the Baldessari catalogue (Sprengel Museum, 1999), the reference run.

The cell is tagged `parameters`, so you can also run the notebook from the command line with [papermill](https://papermill.readthedocs.io/):

```bash
papermill catalogue-ocr-llm-extraction.ipynb my-catalogue-run.ipynb \
  -p catalogue_pdf /path/to/my-catalogue.pdf -p artist_name "" -p artist_qid "" \
  -p exhibition_title "…" -p exhibition_location "…"
```

In [ ]:
# ── The catalogue ────────────────────────────────────────────────────────────
catalogue_pdf = "baldessari-sprengel.pdf"   # path to the scanned catalogue
output_dir    = ""            # "" = a folder "<pdf name>-output" next to the PDF
ocr_languages = "eng+deu"     # Tesseract language codes, joined with "+" (e.g. "eng", "deu+fra")
pdf_dpi       = 300
catalogue_pages = ""          # PDF pages that hold the catalogue entries, e.g. "27-61" or "27-61, 63";
                              # "" = all pages. Leaving out essays, bibliographies and appendices
                              # avoids picking up illustration captions and book/video lists as entries.

# ── Artist ───────────────────────────────────────────────────────────────────
# Solo show: the artist's name and Wikidata QID, used for every row.
# Group show: set both to "" and the artist is read from each entry and
# matched on Wikidata by name (those matches are marked [?] for checking).
artist_name = "John Baldessari"
artist_qid  = "Q683378"

# ── Exhibition (the same on every row) ───────────────────────────────────────
exhibition_title        = "While something is happening here, something else is happening there: Works 1988–1999"
exhibition_qid          = "Q141747732"   # Wikidata item for the exhibition, or ""
exhibition_location     = "Sprengel Museum Hannover"
exhibition_location_qid = "Q510144"
exhibition_start        = "1999-09-26"   # YYYY-MM-DD or YYYY
exhibition_end          = "2000-01-02"
exhibition_organizer    = ""

# ── LLM ──────────────────────────────────────────────────────────────────────
ollama_model  = "qwen2.5:14b"
ollama_url    = "http://localhost:11434"
chunk_words   = 800      # words per chunk sent to the LLM
overlap_words = 100      # words repeated between chunks, so entries on a boundary aren't cut
max_chunks    = None     # e.g. 5 for a quick test run; None = whole catalogue

# ── Wikidata ─────────────────────────────────────────────────────────────────
use_wikidata = True      # look up artist details (cached, so re-runs work offline)

# ── Review ───────────────────────────────────────────────────────────────────
open_review_app = False  # True = open the review app (section 11) when the run finishes

## Check the setup

Stops with a list of what's missing and how to install it. The spell checker is optional: without it the OCR quality score is skipped.

In [ ]:
import json
import shutil
import subprocess
import urllib.request

problems = []

def need(binary, how):
    if shutil.which(binary):
        return True
    problems.append(f"{binary} not found: {how}")
    return False

need("pdftoppm", "Linux: sudo apt install poppler-utils · macOS: brew install poppler")
if need("tesseract", "Linux: sudo apt install tesseract-ocr · macOS: brew install tesseract tesseract-lang"):
    installed = subprocess.run(["tesseract", "--list-langs"], capture_output=True, text=True).stdout.split()[1:]
    missing_langs = [lang for lang in ocr_languages.split("+") if lang not in installed]
    if missing_langs:   # only fatal if OCR has to run (checked in section 1)
        print(f"WARNING: Tesseract language data missing for {missing_langs}: "
              f"Linux: sudo apt install {' '.join('tesseract-ocr-' + l for l in missing_langs)} · macOS: brew install tesseract-lang")

try:
    import bs4, lxml  # noqa: F401
except ImportError:
    problems.append("Python packages missing: pip install beautifulsoup4 lxml")

try:
    with urllib.request.urlopen(f"{ollama_url}/api/tags", timeout=10) as r:
        models = {m["name"] for m in json.load(r)["models"]}
    if ollama_model not in models and f"{ollama_model}:latest" not in models:
        problems.append(f"Ollama model '{ollama_model}' not downloaded: ollama pull {ollama_model}")
except OSError:
    problems.append(f"Ollama is not running at {ollama_url}: install it from https://ollama.com and start it")

if problems:
    raise RuntimeError("Setup incomplete:\n  - " + "\n  - ".join(problems))

have_aspell = bool(shutil.which("aspell"))
print("Setup OK." + ("" if have_aspell else " (aspell not found: the OCR quality score will be skipped)"))

## Paths and shared code

Output files go in `output_dir`. The data-model columns and the validator are imported from `scripts/` in this repository, so this notebook and `catalogue-ocr.ipynb` always produce the same columns.

In [ ]:
import csv
import difflib
import hashlib
import re
import sys
import urllib.parse
from pathlib import Path

import warnings
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)   # hOCR is XHTML; lxml's HTML parser is intended

pdf_path = Path(catalogue_pdf).expanduser().resolve()
if not pdf_path.exists():
    raise FileNotFoundError(f"Catalogue PDF not found: {pdf_path}")
stem = pdf_path.stem
out  = Path(output_dir).expanduser().resolve() if output_dir else pdf_path.with_name(f"{stem}-output")
out.mkdir(parents=True, exist_ok=True)

PAGES_DIR     = out / "pages"
OCR_TXT       = out / f"{stem}.txt"
OCR_HOCR      = out / f"{stem}.hocr"
LLM_LOG       = out / f"{stem}-llm-log.json"          # every chunk, its pages and the raw LLM reply
OUT_CSV       = out / f"{stem}-artworks.csv"          # data-model CSV (main output)
OUT_QS        = out / f"{stem}-quickstatements.csv"   # Wikidata P-number columns
WIKIDATA_CACHE = out / "wikidata-cache.json"

def find_repo_root():
    for d in [Path.cwd(), *Path.cwd().parents]:
        if (d / "scripts" / "extract_artworks.py").exists():
            return d
    raise FileNotFoundError("scripts/extract_artworks.py not found: open this notebook from inside the linked-open-exhibition repository")

REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "scripts"))
from extract_artworks import CSV_COLUMNS  # noqa: E402
from validate_csv import validate         # noqa: E402

if stem != "baldessari-sprengel" and exhibition_qid == "Q141747732":
    print("WARNING: the exhibition parameters are still the Baldessari example. Update them for this catalogue.\n")

print(f"Catalogue : {pdf_path}")
print(f"Output    : {out}")

## 1. OCR

Renders each PDF page to an image and runs Tesseract once over all of them, writing `<name>.txt` and `<name>.hocr`. **Skipped if both files are already in the output folder.** To reuse OCR from elsewhere, copy them in. To redo the OCR, delete them. A long catalogue can take several minutes.

In [ ]:
def page_number(png):
    return int(re.search(r"-(\d+)\.png$", png.name).group(1))

if OCR_TXT.exists() and OCR_HOCR.exists():
    print(f"Reusing existing OCR: {OCR_TXT.name}, {OCR_HOCR.name}")
else:
    if missing_langs:
        raise RuntimeError(f"Tesseract language data missing for {missing_langs}: see the setup cell for how to install it")
    PAGES_DIR.mkdir(exist_ok=True)
    pngs = sorted(PAGES_DIR.glob("page-*.png"), key=page_number)
    if not pngs:
        print(f"Rendering pages at {pdf_dpi} dpi…")
        subprocess.run(["pdftoppm", "-r", str(pdf_dpi), "-png", str(pdf_path), str(PAGES_DIR / "page")], check=True)
        pngs = sorted(PAGES_DIR.glob("page-*.png"), key=page_number)
    filelist = PAGES_DIR / "filelist.txt"
    filelist.write_text("\n".join(str(p) for p in pngs), encoding="utf-8")

    print(f"Running Tesseract ({ocr_languages}) on {len(pngs)} pages…")
    r = subprocess.run(["tesseract", str(filelist), str(out / stem), "-l", ocr_languages, "txt", "hocr"],
                       capture_output=True, text=True)
    if not (OCR_TXT.exists() and OCR_HOCR.exists()):
        raise RuntimeError(f"Tesseract failed:\n{r.stderr[-2000:]}")

print(f"{OCR_TXT.name}: {OCR_TXT.stat().st_size:,} bytes · {OCR_HOCR.name}: {OCR_HOCR.stat().st_size:,} bytes")

## 2. OCR quality

The share of words that a spell checker for any of the OCR languages recognises. Names, titles and numbers lower the score, so even good OCR rarely reaches 1.0. As a rough guide, at **0.80 or above** the LLM copes well. Below that, look at the page images: the scan may need a higher dpi, or a different OCR engine. This step is informational and never stops the run.

In [ ]:
ASPELL_CODES = {"eng": "en", "deu": "de", "fra": "fr", "ita": "it", "spa": "es", "nld": "nl",
                "por": "pt", "dan": "da", "swe": "sv", "nor": "nb", "pol": "pl", "ces": "cs"}

def ocr_quality():
    words = re.findall(r"[^\W\d_]+(?:['-][^\W\d_]+)*", OCR_TXT.read_text(encoding="utf-8", errors="replace"))
    if not words:
        print("No words found in the OCR text: check the page images.")
        return None
    unknown_per_lang = []
    for lang in ocr_languages.split("+"):
        code = ASPELL_CODES.get(lang)
        if not code:
            continue
        r = subprocess.run(["aspell", "list", f"--lang={code}"], input="\n".join(words),
                           capture_output=True, text=True)
        if r.returncode == 0:
            unknown_per_lang.append(set(r.stdout.split()))
        else:
            print(f"  aspell dictionary '{code}' not installed: skipped")
    if not unknown_per_lang:
        print("No aspell dictionary available: quality score skipped.")
        return None
    unknown = set.intersection(*unknown_per_lang)          # unknown in every language
    score = sum(w not in unknown for w in words) / len(words)
    print(f"{len(words):,} words · quality score {score:.2f} "
          f"({'good' if score >= 0.80 else 'low: check the scan before trusting the results'})")
    return score

quality_score = ocr_quality() if have_aspell else None

## 3. Text and chunks

Reads the hOCR in Tesseract's own reading order (block, then paragraph, then line), which keeps multi-column pages in the right order. Line breaks are kept, because they help the LLM see where one entry ends and the next begins. The text is then cut into chunks of about `chunk_words` words, each overlapping the previous one by `overlap_words` words. Each chunk records which PDF pages it covers.

In [ ]:
LINE_CLASSES = ["ocr_line", "ocr_caption", "ocr_header", "ocr_textfloat"]

def read_hocr():
    """Returns {page number: [paragraph, …]}, each paragraph a list of lines."""
    soup = BeautifulSoup(OCR_HOCR.read_bytes(), "lxml")
    pages = {}
    for page_no, page in enumerate(soup.find_all("div", class_="ocr_page"), start=1):
        paragraphs = []
        for par in page.find_all("p", class_="ocr_par"):
            lines = []
            for line in par.find_all("span", class_=LINE_CLASSES):
                text = " ".join(w.get_text(strip=True) for w in line.find_all("span", class_="ocrx_word")).strip()
                if text:
                    lines.append(text)
            if lines:
                paragraphs.append(lines)
        pages[page_no] = paragraphs
    return pages

def make_chunks(pages, size, overlap):
    """Groups lines into chunks of about `size` words. Returns [{"text", "pages"}]."""
    units = []                                  # (page, line); "" marks a paragraph break
    for page_no, paragraphs in pages.items():
        units.append((page_no, f"[page {page_no}]"))
        for lines in paragraphs:
            units += [(page_no, line) for line in lines] + [(page_no, "")]
    chunks, current, n_words = [], [], 0
    for unit in units:
        current.append(unit)
        n_words += len(unit[1].split())
        if n_words >= size:
            chunks.append(current)
            tail, tail_words = [], 0             # carry the last `overlap` words into the next chunk
            for u in reversed(current):
                if tail_words >= overlap:
                    break
                tail.insert(0, u)
                tail_words += len(u[1].split())
            current, n_words = tail, tail_words
    if any(line for _, line in current):
        chunks.append(current)
    return [{"text": re.sub(r"\n{3,}", "\n\n", "\n".join(line for _, line in c)).strip(),
             "pages": sorted({p for p, _ in c})} for c in chunks]

def parse_pages(spec, last):
    wanted = set()
    for part in filter(None, (x.strip() for x in str(spec).split(","))):
        a, _, b = part.partition("-")
        wanted.update(range(int(a), int(b or a) + 1))
    return wanted or set(range(1, last + 1))

pages = read_hocr()
page_texts = {p: "\n".join("\n".join(lines) for lines in paras) for p, paras in pages.items()}
selected = parse_pages(catalogue_pages, max(pages))
chunks = make_chunks({p: v for p, v in pages.items() if p in selected}, chunk_words, overlap_words)
print(f"{len(pages)} pages ({len(selected & set(pages))} selected) · "
      f"{sum(len(page_texts[p].split()) for p in selected & set(pages)):,} words · {len(chunks)} chunks")
if max_chunks:
    chunks = chunks[:max_chunks]
    print(f"Test run: only the first {len(chunks)} chunks")

## 4. Extract entries with the LLM

Each chunk goes to the model with the instructions below. The model replies in JSON, with temperature 0, so re-runs give the same result. Everything is logged to `<name>-llm-log.json`: the chunk text, its pages, the raw reply and the parsed entries. Chunks that already succeeded with the same model and prompt are reused, so an interrupted run picks up where it stopped. **Delete the log to force a full re-run.**

If entries are being missed, look in the log for the chunk that contains them before changing the prompt.

In [ ]:
artist_hint = (f"The catalogue is for a solo exhibition of {artist_name}; leave artist null unless an entry names someone else. "
               if artist_name else "The catalogue covers several artists; give the artist named for each entry. ")

PROMPT = f"""You read OCR text from a printed exhibition catalogue and find the catalogue entries in it.
A catalogue entry describes one exhibited artwork and lists some of: catalogue number, artist, title, date,
technique or materials, dimensions, lender or collection, inventory number.
{artist_hint}
Ignore essays, interviews, biographies, bibliographies, exhibition histories and photo credits. A title that is
only mentioned in running text is not an entry. The text has OCR errors; correct obvious ones in the values you
return, but never invent information that is not in the text.

Reply with JSON only, in this form: {{"entries": [ ... ]}} (an empty list if there are no entries).
Each entry is an object with these keys (use null when the catalogue doesn't give the value):
  "catalogue_number": string, as printed
  "artist": string
  "title": string, in the language printed (if the title is given in two languages, use the first)
  "title_language": ISO 639-1 code of the title, e.g. "en", "de"
  "date": string, year or range as printed, e.g. "1994" or "1990-1992"
  "object_type": string in English, e.g. "painting", "photograph", "print", "sculpture", "installation", "video"
  "materials": list of strings, as printed, e.g. ["Fotografie", "Acrylfarbe", "Leinwand"]
  "height_cm", "width_cm", "depth_cm": numbers in centimetres (first dimension is height unless stated otherwise)
  "collection": string, the owner or lender line, e.g. "Courtesy Sonnabend Gallery, New York"
  "inventory_number": string
"""

PROMPT_ID = hashlib.sha1(f"{ollama_model}\n{PROMPT}".encode()).hexdigest()[:12]


def ask_llm(text):
    body = {"model": ollama_model, "system": PROMPT, "prompt": text, "stream": False, "format": "json",
            "options": {"temperature": 0, "num_ctx": 8192}}
    req = urllib.request.Request(f"{ollama_url}/api/generate", data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=900) as r:
        return json.load(r)["response"]


def parse_reply(reply):
    data = json.loads(reply)
    if isinstance(data, dict):
        data = data.get("entries", next((v for v in data.values() if isinstance(v, list)), []))
    if not isinstance(data, list):
        raise ValueError("reply is not a list of entries")
    return [e for e in data if isinstance(e, dict)]


log_by_key = {}
if LLM_LOG.exists():
    log_by_key = {e["key"]: e for e in json.loads(LLM_LOG.read_text(encoding="utf-8")) if e.get("status") == "ok"}

llm_log = []
for i, chunk in enumerate(chunks, start=1):
    key = f"{PROMPT_ID}-{hashlib.sha1(chunk['text'].encode()).hexdigest()[:12]}"
    entry = log_by_key.get(key)
    if entry:
        status = "reused"
    else:
        entry = {"key": key, "chunk": i, "pages": chunk["pages"], "input": chunk["text"],
                 "status": "", "reply": "", "entries": []}
        try:
            entry["reply"] = ask_llm(chunk["text"])
            entry["entries"] = parse_reply(entry["reply"])
            entry["status"] = "ok"
        except Exception as e:  # keep going; failed chunks are retried on the next run
            entry["status"] = f"failed: {e}"
        status = entry["status"]
    entry["chunk"] = i
    llm_log.append(entry)
    LLM_LOG.write_text(json.dumps(llm_log, ensure_ascii=False, indent=1), encoding="utf-8")
    print(f"  chunk {i}/{len(chunks)} (pages {chunk['pages'][0]}–{chunk['pages'][-1]}): "
          f"{status}, {len(entry['entries'])} entries")

failed = [e["chunk"] for e in llm_log if not e["status"].startswith("ok")]
raw_entries = [dict(e, _chunk=log["chunk"]) for log in llm_log for e in log["entries"]]
print(f"\n{len(raw_entries)} entries from {len(llm_log)} chunks" +
      (f" · {len(failed)} chunks FAILED: {failed} (run this cell again to retry)" if failed else ""))

## 5. Clean up and merge duplicates

Values are normalised: numbers become numbers, and materials become lists. Because chunks overlap, the same entry is often extracted twice, sometimes once complete and once cut off. Two entries are merged only if they come from neighbouring chunks, have the same title, and **don't disagree on catalogue number, date or dimensions**. That way, different works with the same title (e.g. several *Untitled*) stay separate.

In [ ]:
FIELDS = ["catalogue_number", "artist", "title", "title_language", "date", "object_type", "materials",
          "height_cm", "width_cm", "depth_cm", "collection", "inventory_number"]


def to_number(v):
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        return float(v)
    m = re.fullmatch(r"\s*(\d+(?:[.,]\d+)?)\s*(?:cm)?\s*", str(v or ""))
    return float(m.group(1).replace(",", ".")) if m else None


def clean(e):
    rec = {"_chunks": {e["_chunk"]}}
    for f in FIELDS:
        v = e.get(f)
        if f in ("height_cm", "width_cm", "depth_cm"):
            v = to_number(v)
        elif f == "materials":
            v = v if isinstance(v, list) else re.split(r"\s*[,;]\s*", v) if isinstance(v, str) else []
            v = [str(m).strip() for m in v if str(m).strip()]
        else:   # flatten line breaks (some models write them as a literal "\\n") and extra spaces
            v = re.sub(r"\s+", " ", str(v).replace("\\n", " ")).strip() if v not in (None, "") else None
            if v and v.lower() in ("null", "none", "n/a", "unknown"):
                v = None
        rec[f] = v
    return rec



def norm(v):
    if isinstance(v, list):
        v = " ".join(map(str, v))
    return re.sub(r"[^0-9a-zà-ÿß]+", " ", str(v).lower()).strip()


def filled(v):
    return v not in (None, "", [])


KEY_FIELDS = ["catalogue_number", "date", "height_cm", "width_cm"]   # differing materials wording etc. doesn't block a merge


def same_work(a, b):
    if norm(a["title"]) != norm(b["title"]):
        return False
    if min(abs(x - y) for x in a["_chunks"] for y in b["_chunks"]) > 1:
        return False
    return all(norm(a[f]) == norm(b[f]) for f in KEY_FIELDS if filled(a[f]) and filled(b[f]))


records = []
for rec in (clean(e) for e in raw_entries):
    if not rec["title"]:
        continue
    twin = next((r for r in records if same_work(r, rec)), None)
    if twin:
        for f in FIELDS:
            if not filled(twin[f]) and filled(rec[f]):
                twin[f] = rec[f]
        twin["_chunks"] |= rec["_chunks"]
    else:
        records.append(rec)

print(f"{len(records)} entries after merging ({len(raw_entries)} extracted)")

## 6. Source page

Finds the PDF page each entry is printed on (`source_page`) by searching for its title in the pages of the chunk it came from. It tries an exact match first, then a fuzzy match on single lines to allow for OCR errors. Titles can also appear in essays, so pages with dimensions on them are preferred. These are **PDF page numbers** (the `page-NN.png` images), not the numbers printed in the catalogue.

In [ ]:
DIMENSION_RE = re.compile(r"\d+[.,]?\d*\s*[x×]\s*\d+|\bcm\b", re.IGNORECASE)


def find_page(title, candidates):
    t = norm(title)
    if len(t) < 3:
        return None
    exact = [p for p in candidates if t in norm(page_texts.get(p, ""))]
    if exact:
        return next((p for p in exact if DIMENSION_RE.search(page_texts[p])), exact[0])
    best, best_score = None, 0.0
    for p in candidates:
        for line in page_texts.get(p, "").splitlines():
            score = difflib.SequenceMatcher(None, t, norm(line)).ratio()
            if score > best_score:
                best, best_score = p, score
    return best if best_score >= 0.75 else None


chunk_pages = {log["chunk"]: log["pages"] for log in llm_log}
for rec in records:
    nearby = sorted({p for c in rec["_chunks"] for p in chunk_pages[c]})
    rec["source_page"] = find_page(rec["title"], nearby) or find_page(rec["title"], sorted(page_texts))

print(f"Source page found for {sum(1 for r in records if r['source_page'])}/{len(records)} entries")

## 7. Artists, object types and materials

**Artists:** in a solo show every row gets the artist from the parameters. In a group show each artist name is searched on Wikidata, and the first match that is a person is used. Those matches are marked `[?]` because a common name can match the wrong person. The Artist data model's details (names, birth/death, nationality, profession, GND and ULAN IDs) then come from the artist's Wikidata item. All lookups are cached in `wikidata-cache.json`.

**Object types and materials** are matched to Wikidata through the small lookup tables below. Add terms that come up in your catalogue: the QuickStatements CSV uses these IDs.

In [ ]:
# Lowercase term → Wikidata QID. Add the terms used in your catalogue (English and the catalogue's language).
OBJECT_TYPE_QIDS = {
    "painting": "Q3305213", "photograph": "Q125191", "photography": "Q125191",
    "gelatin silver print": "Q64029133", "video": "Q34508", "drawing": "Q93184",
    "print": "Q11060274", "lithograph": "Q15123870", "monotype": "Q22669635",
    "collage": "Q22669857", "mixed media": "Q1902763", "sculpture": "Q860861",
    "installation": "Q20437094", "watercolor": "Q18761202", "textile": "Q22075301",
}
MATERIAL_QIDS = {
    "acrylic": "Q207849", "acrylic paint": "Q207849", "acrylfarbe": "Q207849",
    "oil": "Q296955", "oil paint": "Q296955", "ölfarbe": "Q296955",
    "canvas": "Q4259259", "leinwand": "Q4259259",
    "gelatin silver": "Q172984", "photograph": "Q125191", "fotografie": "Q125191",
    "photographic emulsion": "Q355549", "photoemulsion": "Q355549", "fotoemulsion": "Q355549",
    "vinyl paint": "Q2135874", "ink": "Q127418", "tinte": "Q127418",
    "pencil": "Q14674", "bleistift": "Q14674", "paper": "Q11472", "papier": "Q11472",
}


def lookup(term, table):
    if not term:
        return None
    t = term.lower().strip()
    if t in table:
        return table[t]
    close = difflib.get_close_matches(t, table, n=1, cutoff=0.85)
    return table[close[0]] if close else None

In [ ]:
WIKIDATA_API = "https://www.wikidata.org/w/api.php"
USER_AGENT = "linked-open-exhibition-catalogue-extraction/1.0 (https://github.com/NFDI4Culture/linked-open-exhibition)"
cache = json.loads(WIKIDATA_CACHE.read_text(encoding="utf-8")) if WIKIDATA_CACHE.exists() else {}
cache.setdefault("search", {})
cache.setdefault("artists", {})


def wikidata(**params):
    url = f"{WIKIDATA_API}?{urllib.parse.urlencode({**params, 'format': 'json'})}"
    with urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": USER_AGENT}), timeout=30) as r:
        return json.load(r)


def claim_values(entity, prop):
    return [s["mainsnak"]["datavalue"]["value"] for s in entity.get("claims", {}).get(prop, [])
            if "datavalue" in s["mainsnak"]]


def search_person(name):
    """QID of the first Wikidata person matching `name`, or ''."""
    hits = [h["id"] for h in wikidata(action="wbsearchentities", search=name, language="en",
                                      type="item", limit=7)["search"]]
    if not hits:
        return ""
    entities = wikidata(action="wbgetentities", ids="|".join(hits), props="claims")["entities"]
    return next((q for q in hits if any(v["id"] == "Q5" for v in claim_values(entities[q], "P31"))), "")


def artist_details(qid):
    entity = wikidata(action="wbgetentities", ids=qid, props="claims")["entities"][qid]
    item_fields = {"artist_given_name": "P735", "artist_family_name": "P734",
                   "artist_nationality": "P27", "artist_profession": "P106"}
    ids = {f: [v["id"] for v in claim_values(entity, p)] for f, p in item_fields.items()}
    all_ids = sorted({q for qs in ids.values() for q in qs})
    labels = {}
    for i in range(0, len(all_ids), 50):
        for q, e in wikidata(action="wbgetentities", ids="|".join(all_ids[i:i + 50]),
                             props="labels", languages="en|de")["entities"].items():
            labels[q] = (e.get("labels", {}).get("en") or e.get("labels", {}).get("de") or {}).get("value", q)

    def date(prop):  # Wikidata precision 11 = day; otherwise keep the year only
        v = claim_values(entity, prop)
        return "" if not v else v[0]["time"].lstrip("+")[:10] if v[0].get("precision", 9) >= 11 else v[0]["time"].lstrip("+")[:4]

    details = {f: "; ".join(labels.get(q, q) for q in qs) for f, qs in ids.items()}
    details.update(artist_birth_year=date("P569"), artist_death_year=date("P570"),
                   artist_gnd_id="; ".join(claim_values(entity, "P227")),
                   artist_ulan_id="; ".join(claim_values(entity, "P245")))
    return details


def tidy_name(name):
    return name.title() if name and name.isupper() else name


for rec in records:
    rec["review"] = []
    if artist_name:
        named = rec["artist"]
        if named and norm(named) not in norm(artist_name) and norm(artist_name) not in norm(named):
            rec["review"].append(f"entry names another artist: {named}")
        rec["artist"], rec["artist_qid"] = artist_name, artist_qid
    else:
        rec["artist"] = tidy_name(rec["artist"]) or ""
        rec["artist_qid"] = ""
        if rec["artist"]:
            if rec["artist"] not in cache["search"] and use_wikidata:
                try:
                    cache["search"][rec["artist"]] = search_person(rec["artist"])
                except OSError as e:
                    print(f"  Wikidata search failed for {rec['artist']}: {e}")
            rec["artist_qid"] = cache["search"].get(rec["artist"], "")
            rec["review"].append(f"artist matched on Wikidata by name ({rec['artist_qid']})" if rec["artist_qid"]
                                 else "artist not found on Wikidata")
        else:
            rec["review"].append("no artist in the entry")
    rec["object_type_qid"] = lookup(rec["object_type"], OBJECT_TYPE_QIDS)
    rec["material_qids"] = [q for q in (lookup(m, MATERIAL_QIDS) for m in rec["materials"]) if q]

for qid in sorted({r["artist_qid"] for r in records if r["artist_qid"]} - set(cache["artists"])):
    if not use_wikidata:
        break
    try:
        cache["artists"][qid] = artist_details(qid)
    except OSError as e:
        print(f"  Wikidata lookup failed for {qid}: {e}")
WIKIDATA_CACHE.write_text(json.dumps(cache, ensure_ascii=False, indent=1), encoding="utf-8")

for rec in records:
    rec["artist_details"] = cache["artists"].get(rec["artist_qid"], {})

artists = sorted({(r["artist"], r["artist_qid"]) for r in records})
print(f"{len(artists)} artist(s): " + ", ".join(f"{n or '?'} ({q or 'no QID'})" for n, q in artists[:15])
      + (" …" if len(artists) > 15 else ""))
print(f"Object types matched: {sum(1 for r in records if r['object_type_qid'])}/{len(records)} · "
      f"entries with ≥1 material matched: {sum(1 for r in records if r['material_qids'])}/{len(records)}")

## 8. Write the CSVs

- **`<name>-artworks.csv`**: the main output, in the data-model columns of `scripts/extract_artworks.py`.
- **`<name>-quickstatements.csv`**: the same entries with Wikidata property columns, for QuickStatements (`CREATE` = new item).

The `notes` column marks rows that need a human check with `[?]`. The validator counts these rows below. Reasons include:
- no date, materials or dimensions, which suggests a title mentioned in the text rather than a real entry
- source page not found
- the same title appears on another page (often an illustration caption repeating a catalogue entry)
- artist matched by name only

In [ ]:
def as_text(v):
    return "" if v is None else f"{v:g}" if isinstance(v, float) else str(v)


def review_notes(rec):
    notes = list(rec["review"])
    if not (rec["date"] or rec["materials"] or rec["height_cm"] or rec["width_cm"]):
        notes.append("no date, materials or dimensions: may be a title mentioned in the text, not an entry")
    if not rec["source_page"]:
        notes.append("source page not found")
    if rec.get("_same_title_pages"):
        notes.append(f"same title also on p. {', '.join(map(str, rec['_same_title_pages']))}: duplicate or a different work?")
    return "[?] " + "; ".join(notes) if notes else ""


def datamodel_row(rec):
    row = {
        "catalogue_no": as_text(rec["catalogue_number"]),
        "artwork_title": rec["title"],
        "artwork_title_lang": as_text(rec["title_language"]),
        "artwork_type": as_text(rec["object_type"]),
        "artwork_inception": as_text(rec["date"]),
        "medium_material": "; ".join(rec["materials"]),
        "height_cm": as_text(rec["height_cm"]),
        "width_cm": as_text(rec["width_cm"]),
        "depth_cm": as_text(rec["depth_cm"]),
        "collection": as_text(rec["collection"]),
        "inventory_number": as_text(rec["inventory_number"]),
        "artist_full_name": rec["artist"],
        "artist_wikidata_qid": rec["artist_qid"],
        **rec["artist_details"],
        "exhibition_title": exhibition_title,
        "exhibition_location": exhibition_location,
        "exhibition_start": exhibition_start,
        "exhibition_end": exhibition_end,
        "exhibition_organizer": exhibition_organizer,
        "exhibition_wikidata_qid": exhibition_qid,
        "source_page": as_text(rec["source_page"]),
        "notes": review_notes(rec),
    }
    unknown = set(row) - set(CSV_COLUMNS)
    if unknown:
        raise KeyError(f"Columns not in the data model (scripts/extract_artworks.py): {sorted(unknown)}")
    return row


def qs_date(date):
    m = re.search(r"\b(\d{4})\b", date or "")
    return f"+{m.group(1)}-00-00T00:00:00Z/9" if m else ""


def qs_row(rec):
    lang = (rec["title_language"] or "und").lower()[:3]
    return {
        "qid": "CREATE",
        "P31": rec["object_type_qid"] or "",
        "P1476": f'{lang}:"{rec["title"]}"',
        "P170": rec["artist_qid"],
        "P571": qs_date(rec["date"]),
        "P186": rec["material_qids"][0] if rec["material_qids"] else "",
        "P2048": as_text(rec["height_cm"]),
        "P2049": as_text(rec["width_cm"]),
        "P2610": as_text(rec["depth_cm"]),
        "P217": as_text(rec["inventory_number"]),
        "P528": as_text(rec["catalogue_number"]),
        "P1431": exhibition_qid,
    }


records.sort(key=lambda r: (r["source_page"] or 10**6, min(r["_chunks"])))
for rec in records:
    rec["_same_title_pages"] = sorted({o["source_page"] or "?" for o in records
                                       if o is not rec and norm(o["title"]) == norm(rec["title"])}, key=str)

with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=CSV_COLUMNS, restval="")
    w.writeheader()
    w.writerows(datamodel_row(r) for r in records)

with open(OUT_QS, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(qs_row(records[0])) if records else ["qid"])
    w.writeheader()
    w.writerows(qs_row(r) for r in records)

print(f"{OUT_CSV.name}: {len(records)} rows")
print(f"{OUT_QS.name}: {len(records)} rows")

## 9. Validate

The same checks as the regex pipeline in `catalogue-ocr.ipynb`: required fields, recommended fields, `[?]` rows and implausible dimensions.

In [ ]:
validate(OUT_CSV)

## 10. Rows to review

Every row marked `[?]`, with its PDF page, so you can check it against the page image in `pages/`. Fix or delete those rows in the CSV before importing.

In [ ]:
to_review = [(r["source_page"], r["title"], review_notes(r)) for r in records if review_notes(r)]
print(f"{len(to_review)} of {len(records)} rows to review"
      + (f" · {len(failed)} LLM chunks failed: re-run section 4" if failed else "") + "\n")
for page, title, note in to_review:
    print(f"  p.{page or '?':<4} {title[:55]:<55} {note[4:]}")
print(f"\nOutputs in {out}")

## 11. Review in the browser (optional)

Opens the review app (`review_app.py`, next to this notebook) on this run's output. The catalogue page is on the left, with each extracted value boxed where it was found. The entry is on the right, in a form you can edit, plus buttons to add or delete entries and download the corrected CSV (`<name>-artworks-reviewed.csv`). Nothing is re-run: the app only reads the files written above.

Set `open_review_app = True` in the parameters, or just run this cell after changing it here. Needs `pip install gradio`. The app appears below the cell. It also prints a local address, which you can open in a full browser tab for more room. To let someone on another computer use it, change `share=False` to `share=True`. That prints a temporary public link (valid 72 hours) that **anyone who has it can open**, scan pages included. Set the environment variable `REVIEW_APP_PASSWORD` before starting Jupyter to require a login (user `demo`).

The app can also be started outside Jupyter: `python review_app.py "<output folder>" --pdf "<catalogue.pdf>"`.


In [ ]:
if open_review_app:
    try:
        import gradio  # noqa: F401
    except ImportError:
        raise RuntimeError("The review app needs Gradio: pip install gradio")
    sys.path.insert(0, str(REPO_ROOT / "catalogues" / "catalogue-extraction"))
    import review_app
    review_app.launch(out, pdf_path, share=False)
else:
    print("Review app not started (set open_review_app = True to open it).")
